# Phase 4 — Boolean Search (AND / OR / NOT)

This notebook builds on the previous phases:

- **Phase 1 (Prepare Data):** `documents.csv.gz` — 44,997 cleaned Stack Overflow questions (`Id`, `Title`, `Body`, `Tags`, `Document`).
- **Phase 2 (Text Preprocessing):** lowercasing → tokenization → stopword removal → lemmatization.
- **Phase 3 (Inverted Index):** `term → postings` dictionary.

Here we implement **Boolean retrieval**: AND as set **intersection**, OR as set **union**, and NOT as **complement** against the complete set of indexed documents.

## 1. Import Libraries

In [ ]:
import os
import re
import glob
import json
import pandas as pd
from collections import Counter
from IPython.display import display

## 2. Load the Document Collection

We use the same `documents.csv.gz` produced in Phase 1, so the Boolean branch and the ranked (TF-IDF) branch search **the same collection**.

In [ ]:
# Look for the documents file on Kaggle first, then in the current folder
candidates = glob.glob("/kaggle/input/**/documents.csv*", recursive=True) + \
             glob.glob("documents.csv*") + glob.glob("dataset/documents.csv*")
candidates = [f for f in candidates if os.path.isfile(f)]

if not candidates:
    raise FileNotFoundError("documents.csv / documents.csv.gz not found. Upload it to Kaggle input or this folder.")

INPUT_PATH = candidates[0]
print("Using file:", INPUT_PATH)

documents_df = pd.read_csv(INPUT_PATH, compression="infer")
documents_df["Document"] = documents_df["Document"].fillna("").astype(str)

print("Shape:", documents_df.shape)
print("Columns:", documents_df.columns.tolist())
display(documents_df[["Id", "Title", "Tags"]].head(3))

## 3. Text Preprocessing (same pipeline as Phase 2)

Queries **must** go through exactly the same preprocessing as the documents. Otherwise the query term `libraries` would never match the indexed term `library`.

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer

for resource in ["wordnet", "omw-1.4"]:
    try:
        nltk.data.find(f"corpora/{resource}")
    except LookupError:
        nltk.download(resource, quiet=True)

lemmatizer = WordNetLemmatizer()

TOKEN_PATTERN = re.compile(
    r"(?ix)c\+\+|c\#|f\#|\.net|[a-z0-9]+(?:[._+-][a-z0-9]+)*"
)

STOPWORDS = {
    "a", "an", "the",
    "am", "is", "are", "was", "were", "be", "been", "being",
    "do", "does", "did",
    "this", "that", "these", "those",
    "i", "me", "my", "we", "our", "you", "your", "he", "she", "they", "them",
    "it", "its",
    "please", "thanks", "thank",
    "very", "just", "also"
}

PROTECTED_TERMS = {
    "not", "no", "nor", "without",
    "if", "else", "for", "while", "in", "on", "as", "from", "with",
    "class", "return", "import", "true", "false", "null", "none"
}
STOPWORDS -= PROTECTED_TERMS


def tokenize(text):
    if not isinstance(text, str):
        return []
    return TOKEN_PATTERN.findall(text.lower())


def lemmatize_token(token):
    # Preserve programming terms (c++, node.js, ...) and protected words
    if not token.isalpha() or token in PROTECTED_TERMS:
        return token
    return lemmatizer.lemmatize(token)


def preprocess(text):
    tokens = [t for t in tokenize(text) if t not in STOPWORDS]
    lemmas = [lemmatize_token(t) for t in tokens]
    return [t for t in lemmas if t not in STOPWORDS]


print(preprocess("How to sort Libraries in Python 3 with C++ and node.js?"))

In [ ]:
documents_df["Tokens"] = documents_df["Document"].apply(preprocess)
print("Documents preprocessed:", len(documents_df))
print("Example tokens:", documents_df.loc[0, "Tokens"][:15])

## 4. Build the Inverted Index (dictionary of sets)

The Phase 3 index stores `[doc_id, tf]` postings, which TF-IDF will need later.
For Boolean search we only need **which documents contain a term**, so each posting list becomes a Python **set** of document IDs. Sets give us intersection, union and difference directly.

In [ ]:
inverted_index = {}

for doc_id, tokens in zip(documents_df["Id"], documents_df["Tokens"]):
    for term in set(tokens):
        inverted_index.setdefault(term, set()).add(int(doc_id))

ALL_DOC_IDS = set(int(d) for d in documents_df["Id"])
id_to_title = dict(zip(documents_df["Id"].astype(int), documents_df["Title"]))

print("Documents (N):", len(ALL_DOC_IDS))
print("Vocabulary size:", len(inverted_index))

print("\nExample postings (word -> indexed term -> document IDs):")
for word in ["python", "pandas", "java", "c++", "node.js"]:
    term = preprocess(word)[0]
    postings = sorted(inverted_index.get(term, set()))
    print(f"{word:8} -> {term:8} -> {len(postings):5} docs | first IDs: {postings[:5]}")

## 5. Boolean Operators

| Operator | Set operation | Python |
|---|---|---|
| `A AND B` | intersection | `A & B` |
| `A OR B` | union | `A \| B` |
| `NOT A` | complement against **all** documents | `ALL_DOC_IDS - A` |

In [ ]:
def boolean_and(a, b):
    return a & b


def boolean_or(a, b):
    return a | b


def boolean_not(a, all_ids=None):
    all_ids = ALL_DOC_IDS if all_ids is None else all_ids
    return all_ids - a

## 6. Query Parser

Rules:

- Operators are written in **UPPERCASE**: `AND`, `OR`, `NOT`. Lowercase `and / or / not` are treated as normal search words (e.g. `not` is a protected term in our index).
- Precedence: `NOT` > `AND` > `OR`, and parentheses `( )` override it.
- Two words without an operator mean `AND` (e.g. `python list` = `python AND list`).
- Every query word goes through the same `preprocess()` as the documents. If one word produces several tokens, they are combined with `AND`.

The parser converts the query to postfix (Shunting-yard algorithm) and evaluates it with a stack of sets.

In [ ]:
OPERATORS = {"NOT": 3, "AND": 2, "OR": 1}
QUERY_PATTERN = re.compile(r"\(|\)|[^\s()]+")


class QueryError(ValueError):
    pass


def split_query(query):
    """Split the raw query into operators, parentheses and terms; add implicit ANDs."""
    raw = QUERY_PATTERN.findall(query)
    items = []
    for tok in raw:
        if tok in OPERATORS or tok in ("(", ")"):
            items.append(tok)
        else:
            items.append(("TERM", tok))

    # Insert implicit AND between: term/')' followed by term/'('/NOT
    result = []
    for item in items:
        if result:
            prev = result[-1]
            prev_is_operand = isinstance(prev, tuple) or prev == ")"
            curr_starts_operand = isinstance(item, tuple) or item in ("(", "NOT")
            if prev_is_operand and curr_starts_operand:
                result.append("AND")
        result.append(item)
    return result


def to_postfix(items):
    """Shunting-yard: infix items -> postfix list."""
    output, stack = [], []
    for item in items:
        if isinstance(item, tuple):
            output.append(item)
        elif item == "(":
            stack.append(item)
        elif item == ")":
            while stack and stack[-1] != "(":
                output.append(stack.pop())
            if not stack:
                raise QueryError("Unbalanced parentheses: extra ')'")
            stack.pop()
        else:  # operator
            while (stack and stack[-1] != "(" and
                   (OPERATORS[stack[-1]] > OPERATORS[item] or
                    (OPERATORS[stack[-1]] == OPERATORS[item] and item != "NOT"))):
                output.append(stack.pop())
            stack.append(item)
    while stack:
        op = stack.pop()
        if op == "(":
            raise QueryError("Unbalanced parentheses: missing ')'")
        output.append(op)
    return output


def term_postings(word, index, notes):
    """Preprocess one query word and return the set of documents containing it."""
    terms = preprocess(word)
    if not terms:
        notes.append(f"'{word}' was removed by preprocessing (stopword) -> matches nothing")
        return set()
    result = None
    for t in terms:
        postings = index.get(t, set())
        result = postings if result is None else result & postings
    if len(terms) > 1:
        notes.append(f"'{word}' -> {terms} (combined with AND)")
    elif terms[0] != word.lower():
        notes.append(f"'{word}' -> '{terms[0]}'")
    return set(result)


def boolean_search(query, index=None, all_ids=None):
    """Return (matching_doc_ids, notes) for a Boolean query."""
    index = inverted_index if index is None else index
    all_ids = ALL_DOC_IDS if all_ids is None else all_ids

    if query is None or not query.strip():
        raise QueryError("Empty query. Please type at least one word.")

    notes = []
    postfix = to_postfix(split_query(query))
    stack = []
    for item in postfix:
        if isinstance(item, tuple):
            stack.append(term_postings(item[1], index, notes))
        elif item == "NOT":
            if not stack:
                raise QueryError("NOT needs a term after it")
            stack.append(all_ids - stack.pop())
        else:
            if len(stack) < 2:
                raise QueryError(f"{item} needs a term on both sides")
            right, left = stack.pop(), stack.pop()
            stack.append(left & right if item == "AND" else left | right)

    if len(stack) != 1:
        raise QueryError("Invalid query")
    return stack[0], notes


print(split_query("python list sorting"))
print(to_postfix(split_query("(react OR angular) AND NOT jquery")))

## 7. Displaying Results

In [ ]:
def show_results(query, top=10):
    print("=" * 80)
    print("Query:", query)
    try:
        results, notes = boolean_search(query)
    except QueryError as e:
        print("Error:", e)
        return set()
    for n in notes:
        print("  note:", n)
    print(f"Matching documents: {len(results)}")
    for doc_id in sorted(results)[:top]:
        print(f"  {doc_id} | {id_to_title[doc_id][:70]}")
    if len(results) > top:
        print(f"  ... and {len(results) - top} more")
    return results

## 8. Verification on the Worked Example from the Guide

Before testing on 44,997 documents, we check the logic on the four documents from the course guide, where the correct answers are known.

In [ ]:
toy_docs = {
    "D1": "Diabetes treatment includes medication and exercise.",
    "D2": "Heart disease affects blood circulation.",
    "D3": "Common diabetes symptoms include thirst and fatigue.",
    "D4": "Cancer treatment may include chemotherapy.",
}
toy_index = {}
for d, text in toy_docs.items():
    for t in set(preprocess(text)):
        toy_index.setdefault(t, set()).add(d)
toy_all = set(toy_docs)

tests = {
    "diabetes AND treatment": {"D1"},
    "diabetes OR cancer": {"D1", "D3", "D4"},
    "diabetes AND NOT treatment": {"D3"},
}
for q, expected in tests.items():
    got, _ = boolean_search(q, toy_index, toy_all)
    print(f"{q:30} -> {sorted(got)}   expected {sorted(expected)}   {'OK' if got == expected else 'FAIL'}")
    assert got == expected

## 9. Test Queries on the Stack Overflow Collection

### 9.1 AND (intersection)

In [ ]:
r_and = show_results("python AND pandas")

### 9.2 OR (union)

In [ ]:
r_or = show_results("kotlin OR swift")

### 9.3 NOT (complement)

In [ ]:
r_not = show_results("python AND NOT django")

### 9.4 Combined query with parentheses

In [ ]:
r_combo = show_results("(react OR angular) AND NOT jquery")
r_prog = show_results("c++ AND pointer")

### 9.5 Correctness Check (brute force)

To prove the results are not hard-coded, we recompute each query by scanning every document's tokens directly and compare with the index-based answer.

In [ ]:
doc_tokens = {int(d): set(t) for d, t in zip(documents_df["Id"], documents_df["Tokens"])}

def has(term):
    return {d for d, toks in doc_tokens.items() if term in toks}

checks = {
    "python AND pandas": has("python") & has("panda"),
    "kotlin OR swift": has("kotlin") | has("swift"),
    "python AND NOT django": has("python") - has("django"),
    "(react OR angular) AND NOT jquery": (has("react") | has("angular")) - has("jquery"),
}
for q, expected in checks.items():
    got, _ = boolean_search(q)
    status = "OK" if got == expected else "FAIL"
    print(f"{q:35} index={len(got):5}  brute force={len(expected):5}  {status}")
    assert got == expected

# Set identities: |A OR B| = |A| + |B| - |A AND B|  and  |A| + |NOT A| = N
a, _ = boolean_search("kotlin"); b, _ = boolean_search("swift")
ab, _ = boolean_search("kotlin AND swift"); not_a, _ = boolean_search("NOT kotlin")
assert len(r_or) == len(a) + len(b) - len(ab)
assert len(a) + len(not_a) == len(ALL_DOC_IDS)
print("Set identity checks passed.")

### 9.6 Edge Cases

In [ ]:
for q in ["", "   ", "python AND", "(python OR java", "NOT", "the AND python", "Libraries AND python"]:
    show_results(q, top=3)

## 10. Boolean Matching vs Ranked Matching

| | Boolean search (this phase) | Ranked search (TF-IDF + cosine) |
|---|---|---|
| Result | An **exact set**: a document either matches or not | A **ranked list** with a similarity score |
| Order | No notion of "more relevant" (we sort by ID) | Sorted by score, best first |
| Query | Needs operators (AND / OR / NOT) | Free text, e.g. `python list sorting error` |
| Partial match | A document missing one AND term is excluded | A document matching some terms still gets a score |
| Term weight | Every term counts the same | Rare terms (high IDF) and frequent terms in the doc (high TF) weigh more |

Example: `python AND pandas` returns every question containing both words, whether `pandas` appears once or twenty times. TF-IDF would put the question that is mostly about pandas at the top.

## 11. Interactive Search (optional)

Type a Boolean query, or `exit` to stop.

In [ ]:
RUN_INTERACTIVE = False  # set to True to try your own queries

while RUN_INTERACTIVE:
    q = input("Boolean query (or 'exit'): ")
    if q.strip().lower() == "exit":
        break
    show_results(q)